# week08 · 01 — 3층 계면 셀 만들기 (석영 | 물 | 톨루엔)

덱 8-1 · 15장(P8-1). 실행: week08 폴더의 Claude Code 에 「`@notebooks/01_stack_cell.ipynb` 를 실행하고 결과를 요약해」(= week08 폴더에서 `python ../run_nb.py notebooks/01_stack_cell.ipynb` · 직접 보려면 JupyterLab ▶ 모두 실행). 실행하면 `out/stack.data` · `out/in.stack` · `out/stack_summary.json` 이 생기고, LAMMPS 가 있으면 500 스텝을 돌려 `out/stack_sideview.json` 까지 만든다.

첫 셀의 매개변수 3개(물 층 두께 · 톨루엔 수 · 진공)만 바꾼다. 수치 출처는 `scripts/stack_cell.py` 머리말.

In [ ]:
WATER_A = 10.0   # 물 층 두께 Å (0 = 건조 계면)
NTOL    = 20     # 톨루엔 분자 수 (초기 밀도 0.2 g/cm³ → 두께 자동)
VAC_A   = 30.0   # 진공층 Å (≥ 30 · 원본 04주차 #39)
import os, subprocess, sys, json, shutil
# 기본 = 로컬 week08 폴더(Claude Code · JupyterLab) → 아래 if 는 건너뜀 · (선택) Colab 등 저장소 밖이면 clone
if not os.path.exists('scripts'):
    subprocess.run(['git','clone','https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git'], check=False); os.chdir([d for d in os.listdir('.') if d.startswith('construction')][0] + '/week08') if any(d.startswith('construction') for d in os.listdir('.')) else None
PACKMOL = shutil.which('packmol') or 'packmol'
LMP = shutil.which('lmp') or 'lmp'
print('packmol', PACKMOL, '| lmp', LMP if os.path.exists(LMP) else '없음 → 셀만 만든다')

In [ ]:
cmd = [sys.executable, 'scripts/stack_cell.py', '--water', str(WATER_A), '--ntol', str(NTOL), '--vac', str(VAC_A), '--packmol', PACKMOL]
if os.path.exists(LMP): cmd += ['--lmp', LMP, '--steps', '500']
r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout[-2500:]); print(r.stderr[-800:])
S = json.load(open('out/stack_summary.json'))
print('셀', S['Lx_A'], S['Ly_A'], S['Lz_A'], '| 원자', S['atoms_total'], '| 변/2rc', S['min_edge_over_2rc'])
for k in ('slab','water','binder'): print(k, S[k])

## 측면 뷰 — 층이 섞이지 않았는가
점 하나 = 원자(x–z 투영) · 회색 = 석영(고정) · 파랑 = 물 · 주황 = 톨루엔. 500 스텝 뒤에도 세 층의 z 범위가 겹치지 않으면 통과.

In [ ]:
import matplotlib.pyplot as plt
if os.path.exists('out/stack_sideview.json'):
    V = json.load(open('out/stack_sideview.json')); ns = S['slab']['atoms']; nw = S['water']['atoms']
    col = ['#6E655C' if a['id'] <= ns else ('#4F7FB0' if a['id'] <= ns + nw else '#E37F1A') for a in V['atoms']]
    plt.figure(figsize=(5,8)); plt.scatter([a['x'] for a in V['atoms']], [a['z'] for a in V['atoms']], s=6, c=col)
    for k, v in V['layers'].items(): plt.axhline(v['zmax'], ls=':', c='k', lw=.6); plt.text(S['Lx_A']+0.5, v['zmean'], f"{k} {v['n']}")
    plt.xlabel('x (Å)'); plt.ylabel('z (Å)'); plt.title('3층 셀 · 500 스텝 뒤'); plt.show(); print(V['layers'])
else: print('LAMMPS 미실행 — expected_output/stack_sideview.json 을 본다')

## 확인
- `stack_summary.json`: `min_edge_over_2rc ≥ 1` (셀 한 변 ≥ 2 × 12 Å) · `slab.charge_sum = 0` · `packmol.success = true`
- 로그 `out/log.stack`: 온도 ~298 K · 층 z 범위 분리 · `WARNING: System is not charge neutral` 의 값이 1e−5 수준(AM1-BCC 반올림)이면 정상
- 못 돌아가면 `expected_output/` 의 같은 이름 파일로 다음 노트북 진행